# Cinematica direta basica - robo snake

Este notebook segue o mesmo passo a passo usado no material do professor para o MH900, mas aplicado ao robo snake.

A ideia e:

1. montar a matriz de transformacao homogenea generica a partir de rotacoes e translacoes;
2. definir a tabela DH do robo;
3. calcular a cinematica direta;
4. obter a posicao final da ponta.


## Foto do robo

A prancha tecnica abaixo reune as vistas do elo curvo e do suporte de articulacao, com as cotas usadas no modelo.

![Prancha tecnica de referencia](imagens/prancha_tecnica_referencia.png)

As principais medidas usadas na cinematica direta sao:

- distancia centro-centro entre juntas do suporte: `15,17 mm`;
- deslocamento horizontal da pose de referencia: `62,39 mm`;
- deslocamento vertical centro-centro da pose de referencia: `115,51 mm`.

Neste notebook, o referencial foi definido com a origem `(0, 0)` no ponto inferior da montagem, isto e, no ponto que apareceu em laranja no grafico anterior. A ponta/fim do robo fica no ponto superior da montagem.


## Importando bibliotecas

In [ ]:
from sympy import *
from IPython.display import display, Markdown
from IPython.core.interactiveshell import InteractiveShell
import matplotlib.pyplot as plt
import numpy as np

InteractiveShell.ast_node_interactivity = "all"
init_printing(use_latex=True)


## 1. Matriz de Transformacao Homogenea Generica obtida a partir dos Rotacionais e Translacionais

Seguindo o mesmo procedimento do material do professor, a matriz homogenea de cada elo pode ser construida por quatro transformacoes:

- `RotZ(theta)`: rotacao em torno do eixo `z`;
- `TranslZ(d)`: translacao no eixo `z`;
- `TranslX(a)`: translacao no eixo `x`;
- `RotX(alpha)`: rotacao em torno do eixo `x`.

Na convencao DH padrao:

`H = RotZ(theta) * TranslZ(d) * TranslX(a) * RotX(alpha)`


In [ ]:
def componentes_matriz(theta, d, a, alpha):
    RotZ = Matrix([
        [cos(theta), -sin(theta), 0, 0],
        [sin(theta),  cos(theta), 0, 0],
        [0,           0,          1, 0],
        [0,           0,          0, 1],
    ])

    TranslZ = Matrix([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, d],
        [0, 0, 0, 1],
    ])

    TranslX = Matrix([
        [1, 0, 0, a],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ])

    RotX = Matrix([
        [1, 0,           0,          0],
        [0, cos(alpha), -sin(alpha), 0],
        [0, sin(alpha),  cos(alpha), 0],
        [0, 0,           0,          1],
    ])

    return RotZ, TranslZ, TranslX, RotX


def matriz_dh(theta, d, a, alpha):
    RotZ, TranslZ, TranslX, RotX = componentes_matriz(theta, d, a, alpha)
    return simplify(RotZ * TranslZ * TranslX * RotX)


### 1.1 Componentes da matriz

Primeiro, definimos as variaveis simbolicas e exibimos separadamente as quatro matrizes.


In [ ]:
theta, d, a, alpha = symbols("theta d a alpha")
RotZ, TranslZ, TranslX, RotX = componentes_matriz(theta, d, a, alpha)

display(Markdown("**RotZ(theta)**"))
display(RotZ)

display(Markdown("**TranslZ(d)**"))
display(TranslZ)

display(Markdown("**TranslX(a)**"))
display(TranslX)

display(Markdown("**RotX(alpha)**"))
display(RotX)


### 1.2 Multiplicacao das componentes

Agora multiplicamos as quatro componentes para obter a matriz de transformacao homogenea generica.


In [ ]:
H_generica = matriz_dh(theta, d, a, alpha)
display(H_generica)


## 2. Dados do robo snake

O robo e formado por:

- 3 modulos;
- 3 juntas por modulo;
- 9 juntas no total.

Da cota do modulo:

- distancia entre centros das juntas: `15,17 mm`.

Convertendo para metros:

`L = 15,17 / 1000 = 0,01517 m`


In [ ]:
n_modulos = 3
juntas_por_modulo = 3
n_juntas = n_modulos * juntas_por_modulo

L_mm = 15.17
L = L_mm / 1000

print("Numero de juntas:", n_juntas)
print("Comprimento entre juntas:", L_mm, "mm")
print("Comprimento entre juntas:", L, "m")


## 3. Tabela de parametros DH do robo snake

Como o primeiro modelo e planar, cada junta gira em torno do eixo `z`.

Para cada elo:

- `theta_i`: angulo da junta;
- `d_i = 0`;
- `a_i = L`;
- `alpha_i = 0`.

Assim, a tabela DH tem 9 linhas, uma para cada junta.


In [ ]:
theta_symbols = symbols(f"theta1:{n_juntas + 1}")

tabela_dh = []
for i in range(n_juntas):
    tabela_dh.append({
        "elo": i + 1,
        "theta": theta_symbols[i],
        "d": 0,
        "a": L,
        "alpha": 0,
    })

for linha in tabela_dh:
    print(
        f"Elo {linha['elo']}: "
        f"theta={linha['theta']}, d={linha['d']}, a={linha['a']}, alpha={linha['alpha']}"
    )


## 4. Matriz de cada elo

Substituindo `d = 0`, `a = L` e `alpha = 0`, cada elo do snake tem a seguinte matriz homogenea:


In [ ]:
H_elo = matriz_dh(theta, 0, L, 0)
display(H_elo)


## 5. Cinematica direta por multiplicacao das matrizes

A transformacao total da base ate a ponta e obtida pela multiplicacao:

`T_total = H1 * H2 * H3 * ... * H9`

Como ha muitas juntas, faremos o calculo numerico usando os angulos escolhidos.


In [ ]:
q_graus = np.array([
     94.61,  17.87,  19.10,
     10.48,  -3.75, -15.83,
    -17.68,  -5.46,  20.42,
])

q = np.deg2rad(q_graus)
print("Angulos em graus:", q_graus)
print("Angulos em radianos:", np.round(q, 4))


In [ ]:
def matriz_dh_numerica(theta, d, a, alpha):
    return np.array([
        [np.cos(theta), -np.sin(theta) * np.cos(alpha),  np.sin(theta) * np.sin(alpha), a * np.cos(theta)],
        [np.sin(theta),  np.cos(theta) * np.cos(alpha), -np.cos(theta) * np.sin(alpha), a * np.sin(theta)],
        [0,              np.sin(alpha),                  np.cos(alpha),                 d],
        [0,              0,                              0,                             1],
    ], dtype=float)

T_total = np.eye(4)
posicoes = [T_total[:2, 3].copy()]

for i, theta_i in enumerate(q, start=1):
    H_i = matriz_dh_numerica(theta_i, 0, L, 0)
    T_total = T_total @ H_i
    posicoes.append(T_total[:2, 3].copy())

posicoes = np.array(posicoes)

display(Markdown("**Matriz final da base ate a ponta**"))
display(Matrix(np.round(T_total, 5)))


## 6. Posicao final da ponta

A posicao da ponta e retirada da ultima coluna da matriz homogenea final.


In [ ]:
ponta = posicoes[-1]

print("Posicao da ponta em metros:", np.round(ponta, 5))
print("Posicao da ponta em milimetros:", np.round(ponta * 1000, 2))

# Como a origem agora esta no ponto inferior, o alvo superior fica a esquerda e acima.
alvo_desenho_mm = np.array([-62.39, 115.51])
erro_mm = ponta * 1000 - alvo_desenho_mm

print("Medida do desenho [mm]:", alvo_desenho_mm)
print("Erro em relacao ao desenho [mm]:", np.round(erro_mm, 4))
print("Erro total [mm]:", round(np.linalg.norm(erro_mm), 4))


## 7. Comparacao com a forma direta por seno e cosseno

Como o robo e planar, a mesma cinematica direta tambem pode ser escrita pela soma dos angulos acumulados:

`phi1 = theta1`

`phi2 = theta1 + theta2`

`phi3 = theta1 + theta2 + theta3`

E assim por diante.


In [ ]:
phi = np.cumsum(q)

x = L * np.sum(np.cos(phi))
y = L * np.sum(np.sin(phi))

print("Resultado pela soma seno/cosseno [mm]:", np.round(np.array([x, y]) * 1000, 2))
print("Resultado pela matriz homogenea [mm]:", np.round(ponta * 1000, 2))


## 8. Desenho do robo

O grafico abaixo mostra a pose calculada pela cinematica direta. Cada cor representa um modulo.


In [ ]:
cores = ["#0b36b8", "#111111", "#3267d6"]

fig, ax = plt.subplots(figsize=(7, 5))

for modulo in range(n_modulos):
    inicio = modulo * juntas_por_modulo
    fim = inicio + juntas_por_modulo
    trecho = posicoes[inicio:fim + 1]
    ax.plot(
        trecho[:, 0] * 1000,
        trecho[:, 1] * 1000,
        "-o",
        linewidth=2.5,
        color=cores[modulo],
        label=f"Modulo {modulo + 1}",
    )

ax.scatter([posicoes[0, 0] * 1000], [posicoes[0, 1] * 1000], s=90, color="#ff7f0e", label="Base / origem")
ax.scatter([posicoes[-1, 0] * 1000], [posicoes[-1, 1] * 1000], s=90, color="#1f77b4", label="Ponta / fim")
ax.scatter([alvo_desenho_mm[0]], [alvo_desenho_mm[1]], marker="x", s=120, color="red", label="Medida do desenho")
ax.set_title("Cinematica direta - robo snake")
ax.set_xlabel("X [mm]")
ax.set_ylabel("Y [mm]")
ax.axis("equal")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## Resumo

Seguindo o procedimento do professor:

1. foram definidas as matrizes `RotZ`, `TranslZ`, `TranslX` e `RotX`;
2. essas matrizes foram multiplicadas para obter a matriz DH generica;
3. foi criada a tabela DH do robo snake com 9 juntas;
4. as matrizes dos 9 elos foram multiplicadas;
5. a posicao da ponta foi obtida pela ultima coluna da matriz homogenea final.
